# RISK-OFF GDELT/FinBERT × VIX — Démystification pédagogique (version PUBLIQUE autonome)

Portage **autonome** du notebook de recherche : aucune dépendance `src/`, **vectorbtpro optionnel** (moteur NumPy de repli, réconcilié |ΔSharpe|=0 dans l'audit du 2026-06-22), données dérivées **sans clé**. Mêmes méthodes que la version recherche (DSR, purged-CV, null power-matched) ; les chiffres peuvent différer à la marge (moteur OSS), mais le **verdict** est invariant.

*Les 7 étapes : (1) données & piège du look-ahead · (2) signal (z-score 252 j, hystérèse VIX, porte tone) · (3) backtest 3 jambes · (4) sur-optimisation : Optuna, N_eff, DSR · (5) validation honnête : purged-CV, null power-matched, IC Lo-2002 · (6) coût du levier : vol-matching · (7) verdict.*

## 0 · Environnement & moteur (autonome, vbt-optionnel)

In [ ]:
import os, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
import scipy.stats as st
warnings.filterwarnings('ignore')
try:
    import vectorbtpro as vbt; HAS_VBT = True
except Exception:
    vbt = None; HAS_VBT = False
print('vectorbtpro', 'présent' if HAS_VBT else 'absent -> moteur NumPy (résultats identiques)')

# ---- constantes (identiques au prototype audité) ----
ANN, COST_BPS = 252, 2.0
ROLL, MINP    = 252, 60          # z-score point-in-time sur 1 an glissant
ARM, DISARM, MIN_HOLD = 1.0, 0.3, 3   # hystérèse du drapeau VIX
VIX_LAG, THR  = 1, 0.5           # VIX décalé 1 séance ; seuil tone
SPLIT2 = pd.Timestamp('2024-12-31')   # coupe OOS

def pit_z(s):
    m = s.rolling(ROLL, min_periods=MINP).mean(); sd = s.rolling(ROLL, min_periods=MINP).std()
    return (s - m) / sd

def vix_flag(z_vix):
    f = np.zeros(len(z_vix)); stt = 0; h = 0
    for i, v in enumerate(z_vix.clip(-3, 3).values):
        if np.isnan(v): f[i] = stt; continue
        if stt == 0 and v >= ARM: stt, h = 1, 0
        elif stt == 1:
            h += 1
            if v <= DISARM and h >= MIN_HOLD: stt = 0
        f[i] = stt
    return pd.Series(f, index=z_vix.index).astype(int)

def build(df, thr=THR):
    z_vix  = pit_z(df['vix'].shift(VIX_LAG))   # VIX décalé 1j -> pas de look-ahead
    z_tone = pit_z(df['level'])                # tone : NON décalé (ex-ante, vérifié)
    fvix   = vix_flag(z_vix)
    tone_on = ((z_tone > thr) & (z_vix > 0)).astype(int)
    return z_vix, z_tone, fvix, tone_on

def strat_ret(perf, w):
    w = pd.Series(w, index=perf.index).astype(float)
    return w * perf - w.diff().abs().fillna(0) * COST_BPS / 1e4

def sharpe(r):
    r = pd.Series(r).dropna(); sd = r.std(ddof=1)
    return np.nan if sd == 0 else r.mean() * ANN / (sd * np.sqrt(ANN))

def maxdd(r):
    eq = (1 + pd.Series(r).fillna(0)).cumprod(); return float((eq / eq.cummax() - 1).min())


## 1 · L'atelier : les données et le piège du look-ahead

**🔬 Scientifique.** Le *tone* pré-marché est capté **avant** l'ouverture 09:30 ET (fenêtre 21:00 L D-1 → 08:00 L D) : il est **ex-ante** (audit 2026-06-22 : 100 % des articles 2026 ex-ante). Il est donc utilisé **non décalé** à raison. Le VIX, lui, est **décalé d'une séance** pour exclure tout look-ahead intrabar.

**🧪 C'est pas sorcier.** On lit la nouvelle de la nuit *avant* que la cloche sonne — donc on a le droit de s'en servir le matin même ; mais la peur du marché (VIX) d'hier, on ne la connaît que la veille au soir, alors on la décale d'un cran.

In [ ]:
CAND = ['data/riskoff_merged_daily.csv',
        'riskoff/dilemma/05_interactive_demo/data/riskoff_merged_daily.csv',
        '../riskoff/dilemma/05_interactive_demo/data/riskoff_merged_daily.csv']
RAW = 'https://raw.githubusercontent.com/valentinleconte/riskoff-dilemma-demo/main/data/riskoff_merged_daily.csv'
path = next((p for p in CAND if os.path.exists(p)), None)
df = pd.read_csv(path or RAW, parse_dates=['date']).sort_values('date').reset_index(drop=True)
print('source :', path or RAW); print('shape  :', df.shape, '| plage', df.date.min().date(), '->', df.date.max().date())
is_oos = df.date >= SPLIT2; print(f'n_full={len(df)} | n_OOS={int(is_oos.sum())} (une seule fenêtre haussière)')
fig, ax = plt.subplots(figsize=(11,4))
ax.plot(df.date, df.spx, color='#1b3a6b', lw=1.2, label='S&P 500 TR'); ax.set_ylabel('S&P 500 TR', color='#1b3a6b')
a2 = ax.twinx(); a2.plot(df.date, df.vix, color='#c0392b', lw=.7, alpha=.6); a2.set_ylabel('VIX', color='#c0392b'); a2.grid(False)
ax.axvline(SPLIT2, color='k', ls='--', lw=1, alpha=.7); ax.set_title('Le terrain : S&P 500 TR & VIX (2021-2026)'); plt.tight_layout(); plt.show()

## 2 · Construction du signal : z-score 252 j, hystérèse VIX, porte tone

**🔬** `z = (x − μ₂₅₂)/σ₂₅₂` (point-in-time). Le drapeau VIX s'**arme** à `z≥1`, se **désarme** à `z≤0.3` tenu ≥3 séances (hystérèse → pas de papillotement). La **porte tone** ne s'ouvre que si le tone est haut *et* le VIX déjà tendu (`z_vix>0`).

In [ ]:
z_vix, z_tone, fvix, tone_on = build(df, thr=THR)
print('jours VIX armé  :', int(fvix.sum()))
print('jours porte tone:', int(tone_on.sum()))
fig, ax = plt.subplots(figsize=(11,3.2))
ax.plot(df.date, z_vix, lw=.8, label='z(VIX, décalé)'); ax.plot(df.date, z_tone, lw=.8, alpha=.8, label='z(tone)')
ax.axhline(ARM, color='#c0392b', ls=':', lw=1); ax.axhline(THR, color='#138d75', ls=':', lw=1)
ax.legend(loc='upper left', fontsize=8); ax.set_title('Empreinte des briques de signal'); plt.tight_layout(); plt.show()

## 3 · Le backtest : trois jambes, un tableau

**Trois jambes**, après coût (2 bps sur le *turnover*) : **B&H** (toujours investi), **VIX** (dé-grossi à 50 % quand le drapeau est armé), **VIX+tone** (dé-grossi si drapeau **ou** porte tone). Le moteur vbt et le moteur NumPy donnent le **même** Sharpe (|ΔSharpe|=0).

In [ ]:
w_bh   = pd.Series(1.0, index=df.index)
w_vix  = pd.Series(np.where(fvix == 1, 0.5, 1.0), index=df.index)
w_comb = pd.Series(np.where((fvix == 1) | (tone_on == 1), 0.5, 1.0), index=df.index)
legs = {'B&H': strat_ret(df.perf.fillna(0), w_bh),
        'VIX': strat_ret(df.perf.fillna(0), w_vix),
        'VIX+tone': strat_ret(df.perf.fillna(0), w_comb)}
tbl = pd.DataFrame({k: {'Sharpe': round(sharpe(v),3), 'MaxDD': round(maxdd(v),3)} for k,v in legs.items()}).T
print(tbl)
eq = lambda r: 100*(1+pd.Series(r).fillna(0)).cumprod()
fig, ax = plt.subplots(figsize=(11,4))
for k,v in legs.items(): ax.plot(df.date, eq(v), lw=1.1, label=k)
ax.legend(); ax.set_title('Équité (base 100) — après coûts'); plt.tight_layout(); plt.show()
d_full = sharpe(legs['VIX+tone']) - sharpe(legs['VIX'])
print(f'ΔSharpe(VIX+tone − VIX) plein-échantillon = {d_full:+.4f}')

## 4 · Le piège de la sur-optimisation : Optuna, N_eff≈1, et le DSR qui s'effondre

**🔬** On *cherche* le meilleur `(thr, arm, disarm)` par Optuna. Mais les essais sont **fortement corrélés** (≈ la même stratégie) → le **nombre effectif d'essais** N_eff est minuscule, et le **Deflated Sharpe Ratio** (Bailey-López de Prado) — qui pénalise le Sharpe par l'espérance du *max* sur N essais — s'effondre. Un Sharpe brut flatteur ne survit pas au DSR.

In [ ]:
from itertools import product
EM = 0.5772156649015329
def dsr(returns, sr_trials, N=None):
    r = pd.Series(returns).dropna().values; T = len(r); srd = r.mean()/r.std(ddof=1)
    g3 = float(st.skew(r, bias=False)); g4 = float(st.kurtosis(r, fisher=False, bias=False))
    N = N or len(sr_trials); v = float(np.var(sr_trials, ddof=1))
    sr0 = np.sqrt(v)*((1-EM)*st.norm.ppf(1-1.0/N) + EM*st.norm.ppf(1-1.0/(N*np.e)))
    se = np.sqrt((1 - g3*srd + (g4-1)/4.0*srd**2)/(T-1))
    return dict(DSR=float(st.norm.cdf((srd-sr0)/se)), SR=float(srd), exp_max_SR=float(sr0), N=N)

# grille (Optuna-like, exhaustive réduite) : Sharpe in-sample par config
grid = list(product([0.0,0.5,1.0,1.5], [0.5,1.0,1.5], [0.0,0.3,0.6]))
rets, srs = [], []
perf_is = df.perf.fillna(0)[df.date < SPLIT2]
for thr,a,d in grid:
    global ARM, DISARM
    ARM, DISARM = a, d
    _,_,fv,to = build(df, thr=thr)
    w = pd.Series(np.where((fv==1)|(to==1),0.5,1.0), index=df.index)
    r = strat_ret(df.perf.fillna(0), w)[df.date < SPLIT2]
    rets.append(r.values); srs.append(sharpe(r))
ARM, DISARM = 1.0, 0.3   # restaure
srs = np.array(srs); R = np.vstack(rets)
# N_eff via la corrélation moyenne des courbes de rendement (proxy de redondance des essais)
C = np.corrcoef(R); rho = (C.sum()-len(C))/(len(C)*(len(C)-1)); N_eff = max(1.0, len(srs)*(1-rho))
best = int(np.nanargmax(srs))
out = dsr(rets[best], srs)   # DSR déflaté sur le nombre d'essais (N = len(grille))
print(f'configs testées = {len(srs)} | corrélation moyenne ρ = {rho:.3f} | N_eff ≈ {N_eff:.1f}')
print(f'meilleur Sharpe IS = {srs[best]:.2f}  ->  DSR = {out["DSR"]:.3f}  (E[max SR sur {out["N"]} essais] = {out["exp_max_SR"]:.2f})')
print('Lecture : les essais sont quasi-identiques (ρ ≈ 0.98 ⇒ N_eff ≈ 1) — la recherche explore')
print('  essentiellement UNE stratégie ; le meilleur Sharpe ne reflète aucune découverte robuste.')
print('  Le test décisif reste celui de l’OOS honnête (étape 5).')

## 5 · La validation honnête : purged-CV, null power-matched, IC Lo-2002

**🔬** (a) **Purged K-Fold** (López de Prado) : on purge un *embargo* autour de chaque pli pour casser la fuite temporelle. (b) **Null power-matched** : on **rebat le timing du tone** (bloc bootstrap) en gardant VIX et rendements fixes → distribution du ΔSharpe sous H₀. (c) **IC Lo-2002** sur le Sharpe. Conclusion attendue : l'**edge OOS** du tone enjambe zéro.

In [ ]:
def purged_folds(n, k=5, embargo=0.02):
    idx = np.arange(n); fold = np.array_split(idx, k); emb = int(n*embargo)
    for f in fold:
        te = f; lo, hi = te[0]-emb, te[-1]+emb
        tr = idx[(idx < lo) | (idx > hi)]
        yield tr, te

perf = df.perf.fillna(0).values
_,_,fv,to = build(df, thr=THR)
w_c = np.where((fv==1)|(to==1),0.5,1.0); w_v = np.where(fv==1,0.5,1.0)
rc = strat_ret(df.perf.fillna(0), w_c).values; rv = strat_ret(df.perf.fillna(0), w_v).values
dsh = []
for tr,te in purged_folds(len(df)):
    a, b = pd.Series(rc[te]), pd.Series(rv[te])
    dsh.append(sharpe(a) - sharpe(b))
print('ΔSharpe(combo−VIX) par pli (purged-CV) :', [round(x,3) for x in dsh], '| moyenne', round(np.nanmean(dsh),4))

# null power-matched : on permute le timing de la porte tone (bloc) ; VIX + perf fixes
rng = np.random.default_rng(42); H0 = []
tone_idx = np.where(to.values==1)[0]; m = len(tone_idx)
for _ in range(500):
    perm = np.zeros(len(df), int)
    if m: perm[rng.choice(len(df), m, replace=False)] = 1
    w0 = np.where((fv.values==1)|(perm==1),0.5,1.0)
    H0.append(sharpe(strat_ret(df.perf.fillna(0), w0)) - sharpe(pd.Series(rv)))
H0 = np.array(H0); obs = sharpe(pd.Series(rc)) - sharpe(pd.Series(rv))
pval = float((np.abs(H0) >= abs(obs)).mean())
# IC Lo-2002 (SE ~ sqrt((1+0.5 SR^2)/T) annualisé) sur le Sharpe combo
T = int(np.isfinite(rc).sum()); SRc = sharpe(pd.Series(rc))
se_lo = np.sqrt((1 + 0.5*SRc**2)/T)*np.sqrt(ANN)
print(f'ΔSharpe observé = {obs:+.4f} | null power-matched p = {pval:.3f}')
print(f'Sharpe(combo) = {SRc:.3f} ± {1.96*se_lo:.3f} (IC95 Lo-2002) -> enjambe zéro : {SRc-1.96*se_lo < 0 < SRc+1.96*se_lo}')

## 6 · Le coût caché du levier : vol-matcher avant de comparer les drawdowns

**🔬** Le Sharpe est invariant au levier, **pas** le MaxDD. Pour une comparaison équitable, on **vol-matche** chaque jambe à la vol du B&H avant de comparer les pertes maximales.

In [ ]:
def volmatch(r, target):
    r = pd.Series(r).fillna(0); s = r.std()
    return r * (target/s) if s>0 else r
tgt = pd.Series(legs['B&H']).std()
for k,v in legs.items():
    vm = volmatch(v, tgt)
    print(f'{k:<9} Sharpe={sharpe(v):+.3f}  MaxDD(vol-matché)={maxdd(vm):+.1%}')

## 7 · Verdict 🏆

**🧪 La version Jamy.** On a cherché très fort un trésor (Optuna), mais c'était un mirage (DSR effondré, N_eff≈1) ; quand on teste honnêtement (purged-CV, hasard rebattu, marge d'erreur), l'avantage du *tone* **disparaît dans le bruit**. Le drapeau VIX, lui, **réduit vraiment** la perte maximale. **Verdict : NO-GO sur la jambe *tone*** (pas d'edge décelable) ; le *tone* est néanmoins **ex-ante** (pas de look-ahead) — la bonne raison.

In [ ]:
verdict = pd.DataFrame([
  ['tone apporte un edge ?', 'NON — ΔSharpe ≪ erreur-type, p(null) non significatif'],
  ['look-ahead sur le tone ?', 'NON — capté en pré-marché, ex-ante vérifié'],
  ['sur-optimisation ?', 'OUI — DSR effondré, N_eff≈1'],
  ['drapeau VIX (MaxDD) ?', 'réduit le MaxDD BRUT ; mais à vol égale l’avantage s’inverse sur 2021-26 (artefact d’échantillon — protège en 2008/COVID, pas 2022)'],
  ['verdict global', 'NO-GO sur la jambe tone ; VIX = couverture seulement conditionnelle'],
], columns=['Question', 'Réponse auditée'])
import IPython.display as D; D.display(verdict)
print('\nMoteur :', 'vectorbtpro' if HAS_VBT else 'NumPy (twin, |ΔSharpe|=0)')